# GSB 5544 — Pandas Fundamentals: Asking Questions of a DataFrame  
**SOLUTION**

Every section starts with a **question**; the pandas tool is just how we answer it. Remember the question and the code follows.

**Data:** `coffee_purchases.csv` — every coffee purchase from a bank statement (2018–2025). One **row** = one purchase.

| Column | Meaning |
|---|---|
| `date` | days since Jan 1, 1970 (we'll fix this!) |
| `description` | bank-statement text |
| `amount` | dollars (negative = money out) |
| `account` | which account paid |
| `month`, `year`, `day_of_week` | when |

In [1]:
import pandas as pd
coffee = pd.read_csv("https://raw.githubusercontent.com/gato365/gsb5544_instructor_learn_prep/main/assignments/Data/coffee_purchases.csv")
coffee.head()

,date,description,amount,account,month,year,day_of_week
0,17826,SQ *LUCY'S COFFEE C 10/21 PURCHASE SAN LUIS OB...,-20.10,Kelley,October,2018,Monday
1,17833,SQ *LUCY'S COFFEE C 10/27 PURCHASE SAN LUIS OB...,-7.43,Eman,October,2018,Monday
2,18050,SQ *LUCY'S COFFEE CO 06/02 PURCHASE San Luis O...,-2.75,Eman,June,2019,Monday
3,18057,SQ *LUCY'S COFFEE CO 06/08 PURCHASE San Luis O...,-2.75,Eman,June,2019,Monday
4,18136,SQ *SCOUT COFFEE 08/27 PURCHASE SAN LUIS OBIS CA,-9.21,Kelley,August,2019,Wednesday


---
## 1. *How big is this data set, and what's in it?*  → `.shape`, `type()`, `.dtypes`

In [2]:
# Q: How many purchases (rows) and variables (columns)?
coffee.shape

(821, 7)

In [3]:
# Q: What kind of object is `coffee`?
type(coffee)

<class 'pandas.core.frame.DataFrame'>

In [4]:
# Q: What kind of values are in each column?
coffee.dtypes

date             int64
description     object
amount         float64
account         object
month           object
year             int64
day_of_week     object
dtype: object

| dtype | means |
|---|---|
| `int64` / `float64` | numbers |
| `object` | text |
| `category` | categorical with fixed levels |
| `datetime64` | a real date |

`date` is `int64` — pandas *thinks* it's a number. **We know better.**

In [5]:
# .info() = shape + dtypes + non-missing counts in one report
coffee.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 821 entries, 0 to 820
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   date         821 non-null    int64  
 1   description  821 non-null    object 
 2   amount       821 non-null    float64
 3   account      821 non-null    object 
 4   month        821 non-null    object 
 5   year         821 non-null    int64  
 6   day_of_week  821 non-null    object 
dtypes: float64(1), int64(2), object(4)
memory usage: 45.0+ KB


---
## 2. *What kind of variable is each column, really?*

The dtype is how the computer **stores** it. **You** decide what it **means**:

| Type | Definition | Test |
|---|---|---|
| **Quantitative** | numbers where arithmetic makes sense | "Is the *average* meaningful?" |
| **Categorical** | labels / groups (levels) — even if stored as numbers | "Are the values group names?" |
| **Date / time** | points in time | "Is it a calendar value?" |

| Column | dtype | By definition |
|---|---|---|
| `date` | int64 | **date/time** |
| `description` | object | text |
| `amount` | float64 | **quantitative** |
| `account`, `month`, `day_of_week` | object | **categorical** |
| `year` | int64 | **categorical** (ordered) — the "average year" means nothing |

Now make the DataFrame **state** these types itself.

In [6]:
# Q: How do I make `date` a real date?  (the integers are days since 1970-01-01)
coffee["date"] = pd.to_datetime(coffee["date"], unit="D")
coffee["date"].head(3)

0   2018-10-22
1   2018-10-29
2   2019-06-03
Name: date, dtype: datetime64[ns]

In [7]:
# Q: How do I declare a categorical variable?
coffee["account"] = coffee["account"].astype("category")

# For ORDERED categories, tell pandas the order of the levels
day_order = ["Monday","Tuesday","Wednesday","Thursday","Friday","Saturday","Sunday"]
coffee["day_of_week"] = pd.Categorical(coffee["day_of_week"], categories=day_order, ordered=True)

coffee.dtypes

date           datetime64[ns]
description            object
amount                float64
account              category
month                  object
year                    int64
day_of_week          category
dtype: object

In [8]:
# Q: Why bother?  Because counts now come out in the right order.
coffee["day_of_week"].value_counts().sort_index()

day_of_week
Monday       373
Tuesday      129
Wednesday     93
Thursday     100
Friday       100
Saturday      15
Sunday        11
Name: count, dtype: int64

In [9]:
# Q: Does the average of `year` mean anything?  Pandas will compute it anyway — YOU have to know it's meaningless.
coffee["year"].mean()

np.float64(2022.3848964677222)

✅ **Check:** Is `year` quantitative or categorical? What question would its average (not) answer?

---
## 3. *Am I holding a table or a single column?*  → DataFrame vs Series

One column name in `[ ]` → **Series** (1-D). A **list** of names in `[[ ]]` → **DataFrame** (2-D).

In [10]:
amt = coffee["amount"]          # Series
print(type(amt))
amt.head(3)

<class 'pandas.core.series.Series'>


0   -20.10
1    -7.43
2    -2.75
Name: amount, dtype: float64

In [11]:
amt_df = coffee[["amount"]]     # DataFrame (even with one column)
print(type(amt_df))
amt_df.head(3)

<class 'pandas.core.frame.DataFrame'>


,amount
0,-20.10
1,-7.43
2,-2.75


✅ **Check:** predict `type(coffee[["account", "amount"]])`, then run it.

In [12]:
type(coffee[["account", "amount"]])

<class 'pandas.core.frame.DataFrame'>

---
## 4. *How do I grab columns, rows, or filtered rows?*  → `[ ]`

| You write | You get |
|---|---|
| `df["col"]` | one column |
| `df[["a","b"]]` | several columns |
| `df[0:5]` | rows 0–4 by position |
| `df[condition]` | rows where condition is True |

In [13]:
# Q: What was spent and when?
coffee[["date", "amount"]].head()

,date,amount
0,2018-10-22,-20.10
1,2018-10-29,-7.43
2,2019-06-03,-2.75
3,2019-06-10,-2.75
4,2019-08-28,-9.21


In [14]:
# Q: What are the first 5 purchases?   (a slice in [ ] picks ROWS)
coffee[0:5]

,date,description,amount,account,month,year,day_of_week
0,2018-10-22,SQ *LUCY'S COFFEE C 10/21 PURCHASE SAN LUIS OB...,-20.10,Kelley,October,2018,Monday
1,2018-10-29,SQ *LUCY'S COFFEE C 10/27 PURCHASE SAN LUIS OB...,-7.43,Eman,October,2018,Monday
2,2019-06-03,SQ *LUCY'S COFFEE CO 06/02 PURCHASE San Luis O...,-2.75,Eman,June,2019,Monday
3,2019-06-10,SQ *LUCY'S COFFEE CO 06/08 PURCHASE San Luis O...,-2.75,Eman,June,2019,Monday
4,2019-08-28,SQ *SCOUT COFFEE 08/27 PURCHASE SAN LUIS OBIS CA,-9.21,Kelley,August,2019,Wednesday


In [15]:
# Q: What were the first 5 amounts?   (column, then rows)
coffee["amount"][0:5]

0   -20.10
1    -7.43
2    -2.75
3    -2.75
4    -9.21
Name: amount, dtype: float64

In [16]:
# Q: Which purchases were over $10?
coffee[coffee["amount"] < -10].head()

,date,description,amount,account,month,year,day_of_week
0,2018-10-22,SQ *LUCY'S COFFEE C 10/21 PURCHASE SAN LUIS OB...,-20.10,Kelley,October,2018,Monday
6,2020-01-10,SQ *SCOUT COFFEE 01/09 PURCHASE San Luis Obis CA,-12.91,Eman,January,2020,Friday
49,2020-09-02,SQ *SCOUT COFFEE 09/01 PURCHASE San Luis Obis CA,-10.50,Eman,September,2020,Wednesday
64,2020-11-16,SQ *KRAKEN AVILA 11/13 PURCHASE AVILA BEACH CA,-11.61,Eman,November,2020,Monday
65,2021-01-25,SQ *SCOUT COFFEE 01/24 PURCHASE San Luis Obis CA,-24.20,Eman,January,2021,Monday


In [17]:
# Q: Which purchases were on the Joint account on a Friday?   (& = and, | = or; parentheses required)
coffee[(coffee["account"] == "Joint") & (coffee["day_of_week"] == "Friday")]

,date,description,amount,account,month,year,day_of_week


✅ **Check:** return `description` and `amount` for every purchase in 2025.

In [18]:
# your answer here


---
## 5. *By LABEL or by POSITION?*  → `.loc` vs `.iloc`

Both take **`[rows, columns]`**. `:` alone means "everything".

| | selects by | slice end |
|---|---|---|
| `.loc` | **label** | **inclusive** |
| `.iloc` | **i**nteger position | **exclusive** |

In [19]:
# Q: What is the amount of the first purchase?
print(coffee.loc[0, "amount"])    # row label 0, column "amount"
print(coffee.iloc[0, 2])          # row position 0, column position 2

-20.1
-20.1


In [20]:
# Q: Rows 0-4, date and amount?
coffee.loc[0:4, ["date", "amount"]]     # includes 4

,date,amount
0,2018-10-22,-20.10
1,2018-10-29,-7.43
2,2019-06-03,-2.75
3,2019-06-10,-2.75
4,2019-08-28,-9.21


In [21]:
coffee.iloc[0:4, [0, 2]]               # excludes 4 -> only 4 rows

,date,amount
0,2018-10-22,-20.10
1,2018-10-29,-7.43
2,2019-06-03,-2.75
3,2019-06-10,-2.75


In [22]:
coffee.loc[5, :]                        # one row, all columns -> a Series

date                                        2019-10-07 00:00:00
description    SQ *VERVE COFFEE RO 10/05 PURCHASE SANTA CRUZ CA
amount                                                    -5.33
account                                                    Eman
month                                                   October
year                                                       2019
day_of_week                                              Monday
Name: 5, dtype: object

In [23]:
coffee.loc[:, "amount":"month"].head(3) # a RANGE of columns by label (only .loc)

,amount,account,month
0,-20.10,Kelley,October
1,-7.43,Eman,October
2,-2.75,Eman,June


In [24]:
coffee.iloc[-3:, :3]                    # last 3 rows, first 3 columns

,date,description,amount
818,2025-08-12,SQ *SCOUT COFFEE 08/11 MOBILE PURCHASE San Lui...,-55.00
819,2025-08-05,SQ *SCOUT COFFEE San Luis Obis CA,-4.95
820,2025-08-05,SQ *SCOUT COFFEE San Luis Obis CA,-6.50


In [25]:
# Q: How much was spent on the Business account?   (.loc takes a condition for rows too)
coffee.loc[coffee["account"] == "Business", ["date", "amount"]]

,date,amount
284,2022-04-05,-8.35
294,2022-05-12,-3.25
350,2022-08-11,-3.00
351,2022-08-11,-7.70
361,2022-08-17,-2.50
363,2022-08-17,-4.75
430,2022-10-05,-2.50
431,2022-10-07,-7.75
443,2022-10-18,-3.00
444,2022-10-19,-8.50


In [26]:
# ⚠️ After filtering, labels ≠ positions
big = coffee[coffee["amount"] < -10]
print(big.iloc[0]["amount"])        # first row by POSITION -> works
try:
    big.loc[0]                      # row LABELED 0 -> was filtered out
except KeyError:
    print("KeyError: no row labeled 0")

-20.1


✅ **Check:** with `.loc`, get `account` and `amount` for rows labeled 10–15. Do it again with `.iloc`. Why does one need `10:16`?

In [27]:
# your answer here


---
## 6. *How do I tell pandas what a column really is?*  → `.astype()`, `.map()`, `pd.cut()`

In [28]:
# Q: How do I stop pandas doing math on `year`?  -> number to string
coffee["year"] = coffee["year"].astype(str)
coffee["year"].head(3) + "!"          # string work now, not arithmetic

0    2018!
1    2018!
2    2019!
Name: year, dtype: object

In [29]:
# Q: What are the levels of a categorical?
coffee["account"].cat.categories

Index(['Business', 'Credit Card', 'Eman', 'Joint', 'Kelley'], dtype='object')

In [30]:
# Q: How do I RENAME levels?
coffee["account"] = coffee["account"].cat.rename_categories({"Eman": "Personal-Eman", "Kelley": "Personal-Kelley"})
coffee["account"].value_counts()

account
Personal-Eman      698
Credit Card         82
Joint               24
Business            15
Personal-Kelley      2
Name: count, dtype: int64

In [31]:
# Q: How do I RECATEGORIZE (collapse levels)?  .map() with a dictionary: old level -> new level
account_map = {"Personal-Eman": "Personal",
               "Personal-Kelley": "Personal",
               "Joint": "Shared",
               "Business": "Shared",
               "Credit Card": "Credit"}
coffee["account_group"] = coffee["account"].map(account_map).astype("category")
coffee["account_group"].value_counts()

account_group
Personal    700
Credit       82
Shared       39
Name: count, dtype: int64

In [32]:
# .map() works for any recode — e.g. weekday vs weekend
day_map = {"Saturday": "weekend", "Sunday": "weekend"}
coffee["day_type"] = coffee["day_of_week"].map(day_map).fillna("weekday").astype("category")
coffee["day_type"].value_counts()

day_type
weekday    795
weekend     26
Name: count, dtype: int64

In [33]:
# Q: How do I turn a QUANTITATIVE variable into groups?  pd.cut: bin edges + labels
coffee["spend"] = coffee["amount"].abs()
coffee["size"] = pd.cut(coffee["spend"], bins=[0, 5, 10, 100], labels=["small", "medium", "large"])
coffee["size"].value_counts()

size
small     458
medium    280
large      83
Name: count, dtype: int64

In [34]:
# The DataFrame now states every variable's type
coffee.dtypes

date             datetime64[ns]
description              object
amount                  float64
account                category
month                    object
year                     object
day_of_week            category
account_group          category
day_type               category
spend                   float64
size                   category
dtype: object

✅ **Check:** use `.map()` to recode `month` into `"Fall"`, `"Winter"`, `"Spring"`, `"Summer"`.

In [35]:
# your answer here


---
## 7. *How do I build a table without a file?*  → `pd.DataFrame({...})`

A **dictionary of lists**: keys = column names, lists = values.

In [36]:
# Q: What does a home-made coffee cost?
home = pd.DataFrame({
    "item":     ["beans", "milk", "filter", "sugar"],
    "cost":     [0.85, 0.30, 0.05, 0.02],
    "category": ["ingredient", "ingredient", "supply", "ingredient"],
})
home["category"] = home["category"].astype("category")
home

,item,cost,category
0,beans,0.85,ingredient
1,milk,0.30,ingredient
2,filter,0.05,supply
3,sugar,0.02,ingredient


In [37]:
# Everything above works immediately
print(home.shape, home.dtypes.tolist())
home.loc[home["category"] == "ingredient", "cost"].sum()

(4, 3) [dtype('O'), dtype('float64'), CategoricalDtype(categories=['ingredient', 'supply'], ordered=False, categories_dtype=object)]


np.float64(1.17)

✅ **Check:** build a 4-row DataFrame of coffee shops: `name`, `rating` (quantitative), `has_wifi` (categorical). Declare the category and print `.dtypes`.

In [38]:
# your answer here


---
## Summary

| Question | Tool |
|---|---|
| How big? What's in it? | `.shape`, `type()`, `.dtypes`, `.info()` |
| What kind of variable, really? | `pd.to_datetime()`, `.astype("category")`, `pd.Categorical(ordered=True)` |
| Table or column? | `df["col"]` (Series) vs `df[["col"]]` (DataFrame) |
| Grab columns / rows / filtered rows? | `[ ]` with a name, list, slice, or condition |
| By label or position? | `.loc[rows, cols]` vs `.iloc[rows, cols]` |
| Change a column's type / levels? | `.astype(str)`, `.cat.rename_categories`, `.map({...})`, `pd.cut` |
| Build a table from scratch? | `pd.DataFrame({...})` |